# Evaluation — English ArXiv

Full evaluation with traditional metrics, abstraction metrics,
and LLM-as-Judge scoring.

In [ ]:
!pip install -e /teamspace/studios/this_studio/sm-sip/sm-sip

In [ ]:
from huggingface_hub import login
login()

## Configuration

In [ ]:
from sm_sip.config import SigExtConfig, InferenceConfig, EvalConfig

sigext_config = SigExtConfig.from_preset("en", "1k-60t")
inference_config = InferenceConfig(lang="en", quantization="8bit", prompt_type="source_aware")
eval_config = EvalConfig(lang="en", judge_model_id="Qwen/Qwen2.5-14B-Instruct")

## Load, Process & Evaluate

In [ ]:
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, load_llm, create_summary_chain, create_judge_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt, get_judge_prompt
from sm_sip.pipelines import run_enhanced_evaluation

test_data = get_test_data(lang="en", num_samples=100)
sigext_model, sigext_tokenizer = load_sigext_model(sigext_config.model_id)
processed_data = preprocess_dataset(test_data, sigext_model, sigext_tokenizer, lang="en")

llm_model, llm_tokenizer, gen_pipe = load_llm(inference_config.llm_model_id, inference_config.quantization)
summary_chain = create_summary_chain(gen_pipe, get_summary_prompt("en", "source_aware"))
judge_chain = create_judge_chain(gen_pipe, get_judge_prompt("unified"))

metrics, samples = run_enhanced_evaluation(processed_data, summary_chain, judge_chain, lang="en")
for key, val in metrics.items():
    print(f"  {key}: {val['mean']:.4f}")

## Save & Cleanup

In [ ]:
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory
from datetime import datetime

save_results({"run_info": {"timestamp": datetime.now().isoformat(), "num_samples": len(samples)}, "metrics": metrics, "samples": samples}, "results/english/eval_enhanced.json")
del llm_model, llm_tokenizer, gen_pipe, sigext_model
clear_gpu_memory()